# Đóng gói index Phase 1 thành dataset

Gom các chỉ mục mà lần chạy Phase 1 đã dựng thành **một dataset gọn, có checksum**,
để phiên sau attach vào dùng lại thay vì tốn GPU dựng lại từ đầu.

**Không cần GPU.** Chỉ đọc–chép–băm, vài phút là xong.

---
## Chuẩn bị

1. **Add Input** → attach output của notebook Phase 1 tối qua (đúng cái ông vẫn attach).
2. **Không cần** GPU, không cần secret. Internet chỉ để clone repo.
3. Chạy lần lượt từ trên xuống.

Notebook 06 không xoá gì khi chạy xong, nên output đó chứa sẵn cả cây
`newsqa_phase1/indexes/` lẫn các file `*_checkpoint.tar`. Cell 3 sẽ ưu tiên cây thư
mục; nếu không thấy thì tự giải nén tar mới nhất.


---
## 1. Cấu hình


In [ ]:
# SHA đủ 40 ký tự — git fetch không nhận SHA rút gọn.
REPO_URL    = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT = '2735bdc62df371cbdf53e9feff6b9d5cc5524a3f'   # hoặc 'main'

OUTPUT_NAME = 'phase1-indexes'   # tên thư mục dataset sinh ra

# Lọc bundle theo chuỗi con trong đường dẫn. None = lấy hết.
# Ví dụ chỉ lấy retriever đã khóa + dense winner:  ONLY = ['bge_m3', 'e5_base']
ONLY = None

MAKE_ZIP = False   # True nếu muốn thêm 1 file .zip (tốn gấp đôi chỗ)


In [ ]:
import json, os, shutil, subprocess, sys, tarfile
from pathlib import Path

WORK    = Path('/kaggle/working')
INPUT   = Path('/kaggle/input')
PROJECT = WORK/'Text-Mining---NewsQA-RAG'
OUTPUT  = WORK/OUTPUT_NAME

if not PROJECT.exists():
    subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT)],check=True)

def checkout(ref):
    if ref == 'main':
        subprocess.run(['git','fetch','origin','main'],cwd=PROJECT,check=True)
        subprocess.run(['git','checkout','--detach','origin/main'],cwd=PROJECT,check=True)
        return
    if subprocess.run(['git','fetch','--depth=1','origin',ref],cwd=PROJECT).returncode != 0:
        print('fetch thẳng SHA không được — tải cả lịch sử rồi checkout')
        subprocess.run(['git','fetch','--unshallow','origin'],cwd=PROJECT)
        subprocess.run(['git','fetch','origin'],cwd=PROJECT,check=True)
    subprocess.run(['git','checkout','--detach',ref],cwd=PROJECT,check=True)

checkout(REPO_COMMIT)
print('repo tại', subprocess.run(['git','rev-parse','--short','HEAD'],cwd=PROJECT,
                                 capture_output=True,text=True).stdout.strip())


---
## 2. Tìm nguồn

Quét input xem có chỉ mục nào không. Mỗi thư mục chứa `index_manifest.json` là một
bundle chỉ mục — đó là cách script nhận diện, nên không phụ thuộc vào layout.


In [ ]:
if not INPUT.exists() or not any(INPUT.iterdir()):
    raise SystemExit('Chưa attach input nào. Add Input → chọn output của notebook Phase 1.')

print('Các input đang attach:')
for d in sorted(INPUT.iterdir()):
    print('  ', d.name)

manifests = [p for p in INPUT.rglob('index_manifest.json')
             if '__pycache__' not in str(p)]
SOURCE = None

if manifests:
    # Lấy tổ tiên chung nông nhất còn chứa hết các bundle.
    SOURCE = Path(os.path.commonpath([str(p.parent) for p in manifests]))
    print(f'\nTìm thấy {len(manifests)} bundle chỉ mục dưới:\n  {SOURCE}')
else:
    tars = sorted(INPUT.rglob('*checkpoint.tar'), key=lambda p: p.stat().st_size, reverse=True)
    if not tars:
        raise SystemExit('Không thấy index_manifest.json lẫn checkpoint tar trong input. '
                         'Có chắc đã attach đúng output của Phase 1 không?')
    print(f'\nKhông thấy cây thư mục index — giải nén {tars[0].name} '
          f'({tars[0].stat().st_size/2**30:.1f} GiB)')
    SOURCE = WORK/'unpacked'
    SOURCE.mkdir(exist_ok=True)
    with tarfile.open(tars[0]) as tar:
        # filter='data' chặn đường dẫn thoát ra ngoài; có từ Python 3.12.
        try:              tar.extractall(SOURCE, filter='data')
        except TypeError: tar.extractall(SOURCE)
    found = list(SOURCE.rglob('index_manifest.json'))
    print(f'  giải nén xong, {len(found)} bundle')
    if not found:
        raise SystemExit('Tar không chứa chỉ mục nào.')


---
## 3. Xem trước — chưa ghi gì cả

Xem có những chỉ mục nào và nặng bao nhiêu **trước khi** quyết định gói hết hay lọc bớt.


In [ ]:
preview = [sys.executable,'scripts/package_index_artifacts.py',
           '--source',str(SOURCE),'--dry-run']
if ONLY: preview += ['--only',*ONLY]      # xem trước đúng thứ sẽ được gói
subprocess.run(preview,cwd=PROJECT,check=True)

print()
print('Quá nặng để upload? Quay lại cell 1, đặt')
print("    ONLY = ['bge_m3', 'e5_base']")
print('rồi chạy lại cell này để xem kích thước sau khi lọc.')


---
## 4. Đóng gói

Chép vào `/kaggle/working/`, sinh `manifest.json` (SHA-256 từng file) và `README.md`.


In [ ]:
cmd = [sys.executable,'scripts/package_index_artifacts.py',
       '--source',str(SOURCE),'--output',str(OUTPUT),'--name',OUTPUT_NAME]
if ONLY:     cmd += ['--only',*ONLY]
if MAKE_ZIP: cmd += ['--zip']
subprocess.run(cmd,cwd=PROJECT,check=True)


---
## 5. Kiểm lại

Băm lại toàn bộ file vừa chép và đối chiếu manifest. Nếu lệch thì bản chép hỏng —
đừng upload.


In [ ]:
import hashlib
manifest = json.loads((OUTPUT/'manifest.json').read_text(encoding='utf-8'))
bad = []
for rec in manifest['files']:
    p = OUTPUT/rec['path']
    if not p.exists() or hashlib.sha256(p.read_bytes()).hexdigest() != rec['sha256']:
        bad.append(rec['path'])
print(f"{manifest['file_count']} file, {manifest['total_bytes']/2**30:.2f} GiB")
print('SAI:', bad if bad else 'không có — mọi checksum đều khớp')
assert not bad, bad
print('\n--- README.md ---\n')
print((OUTPUT/'README.md').read_text(encoding='utf-8'))


---
## 6. Xuất bản

### Cách A — dataset Kaggle (đơn giản nhất)

Không cần làm gì thêm. Bấm **Save Version** → toàn bộ `/kaggle/working` thành output
của notebook này, và phiên sau **Add Input** là dùng được ngay.

Muốn thành dataset độc lập có tên riêng: tab **Data** bên phải → **New Dataset** →
trỏ vào thư mục vừa tạo.

### Cách B — Hugging Face

Thêm secret `HF_TOKEN` (Add-ons → Secrets), điền `HF_REPO` rồi chạy cell dưới.

> **Để `private=True`.** Chỉ mục sparse là file **pickle** — giải pickle là thực thi mã
> tuỳ ý. Public hoá nghĩa là bất kỳ ai tải về rồi nạp cũng gánh rủi ro đó, và ta không
> kiểm soát được bản sao nào nữa. README trong gói đã ghi cảnh báo này.


In [ ]:
HF_REPO = ''        # ví dụ 'ThomasAnderson2009/newsqa-phase1-indexes'. Rỗng = bỏ qua.
HF_PRIVATE = True   # giữ nguyên True (xem cảnh báo pickle ở trên)

if not HF_REPO:
    print('Bỏ qua HF. Điền HF_REPO nếu muốn upload.')
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    assert token, 'Chưa có secret HF_TOKEN'
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(HF_REPO, repo_type='dataset', private=HF_PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=str(OUTPUT), repo_id=HF_REPO, repo_type='dataset')
    print(f'Đã upload → https://huggingface.co/datasets/{HF_REPO}')
    print('private =', HF_PRIVATE)


---
## 7. Lần sau dùng lại thế nào

Attach dataset này làm input, rồi trỏ pipeline vào đó thay vì dựng lại. Mỗi bundle giữ
nguyên `index_manifest.json`, `config_*.yaml`, `variant_*.json` của nó nên các đường dẫn
bên trong vẫn nhất quán.

**Luôn kiểm checksum trước khi nạp** — đoạn code có sẵn trong `README.md` của gói. Một
chỉ mục không khớp manifest thì không phải chỉ mục đã được dựng ra, và với file pickle
thì đó không chỉ là chuyện sai số liệu.
